# Quickstart: the traffic-signal MDP, baselines and DQN

This notebook walks through the whole pipeline on a small budget (about 5 minutes on a laptop CPU):

1. the environment, through its Gymnasium API,
2. rule-based baselines,
3. a short DQN training run,
4. a paired comparison on held-out traffic,
5. writing your own controller.

Requires SUMO with `SUMO_HOME` set, and `pip install -e .` from the project root. Run the cells top to bottom.

In [ ]:
import os
import sys
from pathlib import Path

# Everything runs from the project root, where the relative paths in settings/ resolve.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Project root:", PROJECT_ROOT)

## 1. The environment

One `step()` is one decision of the traffic light: pick one of four green phases. If it differs from the current phase, 4 s of yellow come first; then the green runs for 10 s. The reward is the decrease in accumulated waiting time of vehicles on the incoming roads during that interval (so it is usually negative). An episode is 5,400 simulated seconds, and the `seed` passed to `reset()` decides the traffic.

In [ ]:
from environment import Environment
from settings import load_testing_settings

eval_settings = load_testing_settings(Path("settings/testing_settings.yaml"))

env = Environment(
    n_cars_generated=eval_settings.n_cars_generated,
    max_steps=eval_settings.max_steps,
    yellow_duration=eval_settings.yellow_duration,
    green_duration=eval_settings.green_duration,
    turn_chance=eval_settings.turn_chance,
    sumocfg_file=eval_settings.sumocfg_file,
)
print("observation space:", env.observation_space)
print("action space:     ", env.action_space)

obs, info = env.reset(seed=0)
for action in [0, 0, 2, 2, 1, 3]:
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"action {action}  t={info['sim_step']:4d} s  reward={reward:8.1f}  "
          f"seconds simulated this step={len(info['stats'])}")
print("info keys:", sorted(info))

The observation has 240 numbers: 8 lane groups x 10 cells x 3 channels (occupancy, mean speed, mean waiting time), cells getting longer away from the stop line. `info` carries what rule-based controllers need (per-lane queue counts, which lanes each phase serves) and the per-second measurements that every reported metric is computed from.

## 2. Baselines

Every controller implements `choose_action(state, info)` and goes through the same episode loop. We evaluate on the first 5 held-out seeds to keep this quick; the real protocol (`python src/evaluate.py ...`) uses 100.

In [ ]:
from evaluate import evaluate
from policy import FixedTimeController, MaxPressureController, RandomController

quick = eval_settings.model_copy(update={"total_episodes": 5})
baselines = {
    "random": RandomController(seed=0),
    "fixed_time-30s": FixedTimeController(30, quick.green_duration),
    "fixed_time-10s": FixedTimeController(10, quick.green_duration),
    "max_pressure": MaxPressureController(),
}
results = {name: evaluate(ctrl, quick) for name, ctrl in baselines.items()}

for name, r in results.items():
    mean_delay = sum(r["total_delays"]) / len(r["total_delays"])
    print(f"{name:16s} mean total delay {mean_delay:10,.0f} vehicle-seconds")

## 3. Train a DQN agent (short run)

20 episodes is far too few for a good policy (the reference run uses 300); it is enough to see the pipeline work. `training_session` seeds everything, so this cell gives the same result every time on the same machine.

In [ ]:
from IPython.display import Image, display

from settings import load_training_settings
from train import training_session

train_settings = load_training_settings(Path("settings/training_settings.yaml"))
train_settings = train_settings.model_copy(update={"total_episodes": 20, "seed": 0})
out = Path("runs/train/notebook")
log = training_session(train_settings, out)
display(Image(filename=str(out / "training_curves.png")))

## 4. Compare on held-out traffic

All controllers saw the same 5 seeds, so we compare them *per seed* (paired), which removes the variation between easy and hard traffic. With 5 episodes and one training run, read these numbers as a demonstration, not a result.

In [ ]:
from agent import Agent
from evaluation_stats import paired_difference

agent = Agent(settings=train_settings, epsilon=0.0, model_path=out)  # greedy
results["dqn (20 episodes)"] = evaluate(agent, quick)

reference = results["fixed_time-30s"]["total_delays"]
for name, r in results.items():
    d = paired_difference(reference, r["total_delays"])
    print(f"{name:18s} delta vs fixed_time-30s: {d.diff.mean:+9,.0f} veh-s "
          f"[{d.diff.low:+,.0f}, {d.diff.high:+,.0f}]  ({d.pct:+.1f}%)")

## 5. Your turn: a new controller

A controller is any class with `reset()` and `choose_action(state, info)`. Here is "longest queue first": serve the phase with the most halted vehicles on its incoming lanes. Max-pressure is the same rule minus the downstream queue.

In [ ]:
class LongestQueueFirst:
    def reset(self):
        pass

    def choose_action(self, state, info):
        queues = info["lane_queues"]
        waiting = {action: sum(queues.get(lane, 0) for lane in lanes.incoming)
                   for action, lanes in info["phase_lanes"].items()}
        return max(waiting, key=waiting.get)

results["longest_queue"] = evaluate(LongestQueueFirst(), quick)
d = paired_difference(reference, results["longest_queue"]["total_delays"])
print(f"longest_queue delta vs fixed_time-30s: {d.diff.mean:+,.0f} veh-s ({d.pct:+.1f}%)")
env.close()

## Next steps

- Full protocol: train with several seeds, evaluate on all 100 held-out seeds, and compare with `comparison/compare_results.py` (see the README).
- Plug in another algorithm: `Environment` is a regular Gymnasium environment, so Double DQN, PPO, or a library such as Stable-Baselines3 can use it directly. Wrap the trained policy in a class with `choose_action(state, info)` to evaluate it with `evaluate()`.
- `docs/extending.md` lists where to change the state, the reward, the demand, and the network.